# Definizione dei valori

In [1]:
import numpy as np
import pandas as pd
from scipy.integrate import quad
from scipy.optimize import minimize_scalar
import itertools
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, accuracy_score
from sklearn.preprocessing import label_binarize
import os
import warnings
import seaborn as sns

In [2]:
Cf_list = [10000,20000,50000]
MTTF_list = [0.5, 1, 2]
beta_list = [1.5, 2, 2.5, 3, 3.5]
CSystPdM_list = [1000, 2000, 4000, 5000, 10000, 25000] #€/year #MTTF_list/2,5,10, 
#lascio 25000 perchè con alfa basso può avere senso nella preventiva
alfa_list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
Cinter_list = [1000, 2000, 4000, 5000, 10000]#MTTF_list/2,5,10 
#eclusi i duplicati e 25000 perchè non ha senso > di Cf
AN_list = [1-(1/880), 1-(1/1760), 1-(1/3520)] #1-AA=1-(1/MTTF*fsc*Nh)
fsc_list = [1] #interven/h
Nh_list = [1760] #h/year
r_list = [0.99]
# Sostituiamo il prodotto cartesiano cieco di H e F con le tue 3 coppie FMEA reali:
detectability_scenarios = [
    {'detectability': 'low',    'H': 0.65, 'F': 0.30},
    {'detectability': 'medium', 'H': 0.85, 'F': 0.10},
    {'detectability': 'high',   'H': 0.95, 'F': 0.05}
]

# Guasto

In [3]:
# 1. Definiamo le combinazioni uniche solo per MTTF e Cf
# Usiamo itertools.product solo su queste due liste
combinazioni_essenziali = list(itertools.product(MTTF_list, Cf_list))

risultati_guasto = []

for mttf, cf in combinazioni_essenziali:
    # Calcolo del rapporto Cf/MTTF (Costo fisso, non c'è ottimizzazione)
    costo_guasto = cf / mttf
    
    risultati_guasto.append({
        'MTTF': mttf,
        'Cf': cf,
        'Costo_CORRECTIVE': round(costo_guasto, 4)
    })

# 2. Creazione del DataFrame df_GUASTO
df_GUASTO = pd.DataFrame(risultati_guasto)

# 3. Stampa del risultato
print("Database Manutenzione a Guasto (df_GUASTO):")
print("-" * 45)
print(df_GUASTO)
print("-" * 45)

Database Manutenzione a Guasto (df_GUASTO):
---------------------------------------------
   MTTF     Cf  Costo_CORRECTIVE
0   0.5  10000           20000.0
1   0.5  20000           40000.0
2   0.5  50000          100000.0
3   1.0  10000           10000.0
4   1.0  20000           20000.0
5   1.0  50000           50000.0
6   2.0  10000            5000.0
7   2.0  20000           10000.0
8   2.0  50000           25000.0
---------------------------------------------


# Preventiva

In [4]:

# --- 1. CARICAMENTO DATI DA CSV (Formato Orizzontale) ---
df_raw = pd.read_csv('tabella_weibull.csv', header=None, index_col=0)
beta_riferimento = df_raw.loc['beta'].values.astype(float)
rapporto_riferimento = df_raw.loc['MTTF/theta'].values.astype(float)

# --- 3. FUNZIONI DI CALCOLO ---
def get_theta(beta_val, mttf_val):
    rapporto = np.interp(beta_val, beta_riferimento, rapporto_riferimento)
    return mttf_val / rapporto

def costo_obiettivo(tp_val, b, th, cf, cint, csys, alfa, dettagli = False):
    if tp_val <= 0: return np.inf #Se il tempo di intervento è zero o negativo (impossibile fisicamente), la funzione restituisce "infinito".
    rtp = np.exp(-(tp_val / th)**b) # Applica la formula di Weibull per sapere qual è la probabilità che il componente sia ancora vivo al tempo $t_p$.
    area, _ = quad(lambda s: np.exp(-(s / th)**b), 0, tp_val) #Calcola l'integrale della funzione di affidabilità da 0 a $t_p$. Matematicamente, questo rappresenta il tempo atteso di funzionamento del componente prima che venga sostituito (o perché si è rotto o perché abbiamo fatto manutenzione)
    if area == 0: return np.inf
    # Calcolo componenti singole
    c_failure = (cf * (1 - rtp)) / area
    c_preventivo = (cint * rtp) / area
    c_monitoraggio = alfa * csys
    costo_totale = c_failure + c_preventivo + c_monitoraggio
    # Se dettagli è True, restituisce la lista delle componenti
    # Se dettagli è False, restituisce solo il totale (per l'ottimizzatore)
    if dettagli:
        return costo_totale, c_failure, c_preventivo, c_monitoraggio
    
    return costo_totale        


# --- 4. ESPLORAZIONE DI TUTTE LE COMBINAZIONI ---
risultati_totali = []

# Creiamo il prodotto cartesiano di tutte le liste
combinazioni = list(itertools.product(
    beta_list, MTTF_list, Cf_list, Cinter_list, CSystPdM_list, alfa_list
))

print(f"Inizio elaborazione di {len(combinazioni)} combinazioni...")

for b, mttf, cf, cint, csys, alfa in combinazioni:
    th = get_theta(b, mttf)
    
    # Ottimizzazione del tp per questa specifica combinazione
    res = minimize_scalar(
        costo_obiettivo, 
        bounds=(1, mttf * 5), 
        args=(b, th, cf, cint, csys, alfa, False), #dettagli=False
        method='bounded'
    )
    
  # Chiamiamo la funzione una seconda volta con dettagli=True sul punto di ottimo
    tot, fail, prev, mon = costo_obiettivo(res.x, b, th, cf, cint, csys, alfa, dettagli=True)  
    # Salviamo tutto il contesto della combinazione + il risultato
    risultati_totali.append({
        'Beta': b,
        'MTTF': mttf,
        'Cf': cf,
        'Cinter': cint,
        'CSystPdM': csys,
        'Alfa': alfa,
        'Theta': th,
        'tp_Ottimo': round(res.x, 2),
        'Costo_Failure_prev': round(fail, 4),
        'Costo_Prevenzione_prev': round(prev, 4),
        'Costo_Monitoraggio_prev': round(mon, 4),        
        'Costo_PREVENTIVE_Totale': round(res.fun, 4)
    })

# --- 5. CREAZIONE DATAFRAME FINALE ---
df_PREVENTIVA = pd.DataFrame(risultati_totali)



print("Elaborazione completata.")
print(f"Dimensioni del database: {df_PREVENTIVA.shape}")

Inizio elaborazione di 13500 combinazioni...
Elaborazione completata.
Dimensioni del database: (13500, 12)


# Predittiva

In [5]:
# --- 2. GENERAZIONE DELLE COMBINAZIONI (Ottimizzato FMEA) ---

# Definiamo direttamente le sole 3 coppie della Detectability che ti interessano
detectability_scenarios = [
    {'detectability': 'low',    'H': 0.65, 'F': 0.30},
    {'detectability': 'medium', 'H': 0.85, 'F': 0.10},
    {'detectability': 'high',   'H': 0.95, 'F': 0.05}
]

# Rimuoviamo H_list e F_list da itertools.product
combinazioni_pdm_base = list(itertools.product(
    Cf_list, Cinter_list, CSystPdM_list, MTTF_list,
    AN_list, fsc_list, Nh_list, r_list
))

risultati_predittiva = []

# Cicliamo sulle combinazioni base e, per ognuna, inseriamo solo i 3 scenari di Detectability
for cf, cint, csys, mttf, an, fsc, nh, r in combinazioni_pdm_base:
    for det in detectability_scenarios:
        h = det['H']
        f = det['F']
        
        # --- 3. APPLICAZIONE DELLA FORMULA (Identica alla tua) ---
        t1 = csys
        t2 = cint * f * an * fsc * nh * (1 - r)
        t3 = (cint * h) / mttf
        t4 = (cf * (1 - h)) / mttf
        
        costo_totale_pred = t1 + t2 + t3 + t4
        
        # Salvataggio nel formato uniformato
        risultati_predittiva.append({
            'Cf': cf,
            'Cinter': cint,
            'CSystPdM': csys,
            'MTTF': mttf,
            'F': f,
            'H': h,
            'AN': an,
            'fsc': fsc,
            'Nh': nh,
            'r': r,
            'Costo_falsi_positivi': t1,
            'Costo_falsi_negativi': t4,
            'Costo_veri_positivi': t3,
            'Costo_PREDICTIVE_Totale': round(costo_totale_pred, 4)
        })

# --- 4. CREAZIONE DATAFRAME ---
df_PREDITTIVA = pd.DataFrame(risultati_predittiva)

# Stampa tabelle

In [6]:
print ("primi 20 risultati della guasto")
print (df_GUASTO.head(20))
print(f"Totale combinazioni guasto: {len(df_GUASTO)}")
print ("primi 20 risultati della preventiva")
print(df_PREVENTIVA.head(20))
print(f"Totale combinazioni preventiva: {len(df_PREVENTIVA)}")
print ("primi 20 risultati della predittiva")
print(df_PREDITTIVA.head(20))
print(f"Totale combinazioni predittiva: {len(df_PREDITTIVA)}")

primi 20 risultati della guasto
   MTTF     Cf  Costo_CORRECTIVE
0   0.5  10000           20000.0
1   0.5  20000           40000.0
2   0.5  50000          100000.0
3   1.0  10000           10000.0
4   1.0  20000           20000.0
5   1.0  50000           50000.0
6   2.0  10000            5000.0
7   2.0  20000           10000.0
8   2.0  50000           25000.0
Totale combinazioni guasto: 9
primi 20 risultati della preventiva
    Beta  MTTF     Cf  Cinter  CSystPdM  Alfa     Theta  tp_Ottimo  \
0    1.5   0.5  10000    1000      1000   0.1  0.553866        1.0   
1    1.5   0.5  10000    1000      1000   0.2  0.553866        1.0   
2    1.5   0.5  10000    1000      1000   0.3  0.553866        1.0   
3    1.5   0.5  10000    1000      1000   0.4  0.553866        1.0   
4    1.5   0.5  10000    1000      1000   0.5  0.553866        1.0   
5    1.5   0.5  10000    1000      1000   0.6  0.553866        1.0   
6    1.5   0.5  10000    1000      1000   0.7  0.553866        1.0   
7    1.5   0

# Esportazione in csv

In [7]:
df_GUASTO.to_csv('manutenzione_guasto.csv', index=False)
df_PREVENTIVA.to_csv('manutenzione_preventiva.csv', index=False)
df_PREDITTIVA.to_csv('manutenzione_predittiva.csv', index=False)

print("Esportazione completata: i file CSV sono stati creati nella cartella del progetto.")

Esportazione completata: i file CSV sono stati creati nella cartella del progetto.


# Esportazione in Excel

In [8]:
with pd.ExcelWriter('Manutenzione_Completa.xlsx') as writer:
    df_GUASTO.to_excel(writer, sheet_name='Guasto', index=False)
    df_PREVENTIVA.to_excel(writer, sheet_name='Preventiva', index=False)
    df_PREDITTIVA.to_excel(writer, sheet_name='Predittiva', index=False)

print("File Excel multi-foglio creato con successo.")

File Excel multi-foglio creato con successo.


## Combinazioni

In [9]:
# --- 1. SUPER MERGE CON RIPETIZIONE AUTOMATICA (Broadcasting) ---

# Passo A: Uniamo Preventiva e Predittiva sulle colonne comuni.
# Questo genera la struttura macro con tutti i parametri (inclusi Beta, H, F, AN, ecc.)
df_CONFRONTO_GLOBALE = pd.merge(
    df_PREDITTIVA, 
    df_PREVENTIVA, 
    on=['Cf', 'Cinter', 'CSystPdM', 'MTTF'], 
    how='inner'
)

# Passo B: Agganciamo il guasto. Pandas riconosce Cf e MTTF e ripeterà 
# il costo del guasto su tutte le righe generate al Passo A.
df_CONFRONTO_GLOBALE = pd.merge(
    df_CONFRONTO_GLOBALE,
    df_GUASTO,
    on=['Cf', 'MTTF'],
    how='inner'
)

# --- 2. CALCOLO DELLA STRATEGIA VINCENTE CASO PER CASO ---
# Usiamo idxmin lungo l'asse delle colonne per trovare la politica con il costo minore
colonne_costi = ['Costo_CORRECTIVE', 'Costo_PREVENTIVE_Totale', 'Costo_PREDICTIVE_Totale']

# idxmin ci restituisce direttamente il nome della colonna che contiene il valore minimo!
df_CONFRONTO_GLOBALE['Strategia_Ottimale'] = df_CONFRONTO_GLOBALE[colonne_costi].idxmin(axis=1)

# Pulizia estetica dei nomi delle strategie nel verdetto finale
df_CONFRONTO_GLOBALE['Strategia_Ottimale'] = df_CONFRONTO_GLOBALE['Strategia_Ottimale'].str.replace('Costo_', '').str.replace('_Totale', '')

# --- 3. SALVATAGGIO ---
df_CONFRONTO_GLOBALE.to_csv('confronto_politiche_globale.csv', index=False)

print("Database globale generato con successo unendo i tuoi 3 file!")
print(f"Dimensioni finali del database (righe, colonne): {df_CONFRONTO_GLOBALE.shape}")

Database globale generato con successo unendo i tuoi 3 file!
Dimensioni finali del database (righe, colonne): (121500, 24)


In [10]:
print ("primi 20 risultati della combinata")
print (df_CONFRONTO_GLOBALE.head(20))
print(f"Totale combinazioni guasto: {len(df_CONFRONTO_GLOBALE)}")

primi 20 risultati della combinata
       Cf  Cinter  CSystPdM  MTTF    F     H        AN  fsc    Nh     r  ...  \
0   10000    1000      1000   0.5  0.3  0.65  0.998864    1  1760  0.99  ...   
1   10000    1000      1000   0.5  0.3  0.65  0.998864    1  1760  0.99  ...   
2   10000    1000      1000   0.5  0.3  0.65  0.998864    1  1760  0.99  ...   
3   10000    1000      1000   0.5  0.3  0.65  0.998864    1  1760  0.99  ...   
4   10000    1000      1000   0.5  0.3  0.65  0.998864    1  1760  0.99  ...   
5   10000    1000      1000   0.5  0.3  0.65  0.998864    1  1760  0.99  ...   
6   10000    1000      1000   0.5  0.3  0.65  0.998864    1  1760  0.99  ...   
7   10000    1000      1000   0.5  0.3  0.65  0.998864    1  1760  0.99  ...   
8   10000    1000      1000   0.5  0.3  0.65  0.998864    1  1760  0.99  ...   
9   10000    1000      1000   0.5  0.3  0.65  0.998864    1  1760  0.99  ...   
10  10000    1000      1000   0.5  0.3  0.65  0.998864    1  1760  0.99  ...   
11  1

## ALBERI DECISIONALI - FMEA

In [11]:
warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------------------------------------------------
# OUTPUT
# ----------------------------------------------------------------------

output_dir = "output_scenari"
os.makedirs(output_dir, exist_ok=True)

excel_path = os.path.join(
    output_dir,
    "Tree_Predictions_Scenarios.xlsx"
)

writer = pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
)


features = ['Cinter', 'CSystPdM', 'Beta', 'Alfa']
target = 'Strategia_Ottimale'

scenari_predictability = {
    'HIGH': {'H': 0.95, 'F': 0.05},
    'MEDIUM': {'H': 0.85, 'F': 0.10},
    'LOW': {'H': 0.65, 'F': 0.30}
}

scenari_severity = {'HIGH': 50000, 'MEDIUM': 20000, 'LOW': 10000}
scenari_occurrence = {'HIGH': 0.5, 'MEDIUM': 1.0, 'LOW': 2.0}

# ----------------------------------------------------------------------
# STRUTTURE
# ----------------------------------------------------------------------

pie_matrix = {
    p: {s: {o: None for o in scenari_occurrence} for s in scenari_severity}
    for p in scenari_predictability
}




print("🚀 Start scenari")



# ----------------------------------------------------------------------
# LOOP PRINCIPALE
# ----------------------------------------------------------------------
models = {}
for name_p, p_val in scenari_predictability.items():
    for name_s, s_val in scenari_severity.items():
        for name_o, o_val in scenari_occurrence.items():

            scenario_id = f"PRED_{name_p}_SEV_{name_s}_OCC_{name_o}"

            df_scenario = df_CONFRONTO_GLOBALE[
                (df_CONFRONTO_GLOBALE['H'] == p_val['H']) &
                (df_CONFRONTO_GLOBALE['F'] == p_val['F']) &
                (df_CONFRONTO_GLOBALE['Cf'] == s_val) &
                (df_CONFRONTO_GLOBALE['MTTF'] == o_val)
            ].copy()

            if df_scenario.empty:
                continue

            X = df_scenario[features]
            y = df_scenario[target]

            clf = DecisionTreeClassifier(max_depth=3, random_state=42)
            clf.fit(X, y)
            models[(name_p, name_s, name_o)] = clf

            
            y_pred = clf.predict(X)
            

            y_prob = clf.predict_proba(X)

            acc = accuracy_score(y, y_pred)
            leaf_ids = clf.apply(X)
            df_export = df_scenario.copy()
            df_export["TREE_PREDICTION"] = y_pred
            df_export["LEAF_ID"] = leaf_ids
            sheet_name = f"{name_p}_{name_s}_{name_o}"
            df_export.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False
            )
            # ==========================================================
            # PIE MATRIX DATA (QUESTA È LA PARTE CHIAVE)
            # ==========================================================

            dist = (
                pd.Series(y_pred)
                .value_counts(normalize=True)
                .reindex(['PREDICTIVE', 'PREVENTIVE', 'CORRECTIVE'], fill_value=0)
                .tolist()
            )

            pie_matrix[name_p][name_s][name_o] = dist



            # ==========================================================
            # ALBERO
            # ==========================================================

            fig, ax = plt.subplots(figsize=(15, 8))

            plot_tree(
                clf,
                feature_names=features,
                class_names=clf.classes_,
                filled=True,
                rounded=True,
                ax=ax
            )

            ax.set_title(f"{scenario_id} | ACC={acc:.2f}")

            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f"TREE_{scenario_id}.png"), dpi=150)
            plt.close()

            # ==========================================================
            # CONFUSION + ROC + IMPORTANCE
            # ==========================================================

            labels = sorted(y.unique())
            cm = confusion_matrix(y, y_pred, labels=labels)

            fig, axes = plt.subplots(1, 3, figsize=(18, 5))

            sns.heatmap(cm, annot=True, fmt="d", ax=axes[0])
            axes[0].set_title("Confusion Matrix")

            axes[1].barh(features, clf.feature_importances_)
            axes[1].set_title("Feature Importance")

            classes = ['CORRECTIVE', 'PREDICTIVE', 'PREVENTIVE']
            y_bin = label_binarize(y, classes=classes)

            for i, c in enumerate(classes):
                if c in clf.classes_:
                    idx = list(clf.classes_).index(c)
                    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, idx])
                    axes[2].plot(fpr, tpr, label=c)

            axes[2].plot([0, 1], [0, 1], 'k--')
            axes[2].legend()
            axes[2].set_title("ROC")

            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f"REPORT_{scenario_id}.png"), dpi=150)
            plt.close()

# ----------------------------------------------------------------------
# MATRICE PIE CHART 3x3 PER PREDICTABILITY
# ----------------------------------------------------------------------

labels = ['PREDICTIVE', 'PREVENTIVE', 'CORRECTIVE']
colors = ['#4CAF50', '#FFC107', '#F44336']

for p in scenari_predictability.keys():

    fig, axes = plt.subplots(3, 3, figsize=(14, 12))

    fig.suptitle(f"Strategy Distribution - Detectability {p}")

    for i, s in enumerate(scenari_severity.keys()):
        for j, o in enumerate(scenari_occurrence.keys()):

            ax = axes[i, j]
            vals = pie_matrix[p][s][o]

            if vals is None or sum(vals) == 0:
                ax.text(0.5, 0.5, "No Data", ha='center')
                ax.axis("off")
                continue

            ax.pie(
                vals,
                colors=colors,
                autopct=lambda x: f"{x:.1f}%" if x > 0 else "",
                startangle=90
            )

            if i == 0:
                ax.set_title(f"OCC {o}")
            if j == 0:
                ax.set_ylabel(f"SEV {s}")

    fig.legend(labels, loc="lower center", ncol=3)

    plt.tight_layout(rect=[0, 0.05, 1, 0.95])

    plt.savefig(os.path.join(output_dir, f"PIE_MATRIX_{p}.png"), dpi=150)
    plt.close()
    
writer.close()
print("✅ Done")

🚀 Start scenari
✅ Done


## Gestione dati file excel

In [12]:
# ----------------------------------------------------------------------
# INPUT / OUTPUT
# ----------------------------------------------------------------------

input_file = "output_scenari/Tree_Predictions_Scenarios.xlsx"
output_file = "output_scenari/Tree_Predictions_Scenarios_with_costs.xlsx"

xls = pd.ExcelFile(input_file)

writer = pd.ExcelWriter(output_file, engine="openpyxl")

# ----------------------------------------------------------------------
# LOOP SU TUTTI I FOGLI
# ----------------------------------------------------------------------

for sheet_name in xls.sheet_names:

    df = pd.read_excel(xls, sheet_name=sheet_name)

    # colonne strategiche
    pred = df["TREE_PREDICTION"]
    true = df["Strategia_Ottimale"]

    savings = []
    savings_pct = []
    extra_cost = []
    extra_cost_pct = []

    # ------------------------------------------------------------------
    # LOOP RIGHE
    # ------------------------------------------------------------------

    for i in range(len(df)):

        p = pred.iloc[i]
        t = true.iloc[i]

        # costi base
        c_corr = df["Costo_CORRECTIVE"].iloc[i]
        c_pred = df["Costo_PREDICTIVE_Totale"].iloc[i]
        c_prev = df["Costo_PREVENTIVE_Totale"].iloc[i]

        # ------------------------------------------------------------------
        # CASO 1: predizione corretta
        # ------------------------------------------------------------------

        if p == t:

            if p == "CORRECTIVE":
                cost_model = c_corr
            elif p == "PREDICTIVE":
                cost_model = c_pred
            elif p == "PREVENTIVE":
                cost_model = c_prev

            s = c_corr - cost_model
            sp = s / c_corr if c_corr != 0 else 0

            savings.append(s)
            savings_pct.append(sp)
            extra_cost.append(0)
            extra_cost_pct.append(0)

        # ------------------------------------------------------------------
        # CASO 2: predizione errata
        # ------------------------------------------------------------------

        else:

            # costo strategia predetta
            if p == "CORRECTIVE":
                cost_pred = c_corr
            elif p == "PREDICTIVE":
                cost_pred = c_pred
            elif p == "PREVENTIVE":
                cost_pred = c_prev

            # costo strategia corretta
            if t == "CORRECTIVE":
                cost_true = c_corr
            elif t == "PREDICTIVE":
                cost_true = c_pred
            elif t == "PREVENTIVE":
                cost_true = c_prev

            ec = cost_pred - cost_true
            ecp = ec / cost_true if cost_true != 0 else 0

            savings.append(0)
            savings_pct.append(0)
            extra_cost.append(ec)
            extra_cost_pct.append(ecp)

    # ------------------------------------------------------------------
    # AGGIUNTA COLONNE
    # ------------------------------------------------------------------

    df["savings"] = savings
    df["savings_percentuale"] = savings_pct
    df["extra_costo"] = extra_cost
    df["extra_costo_percentuale"] = extra_cost_pct

    # salva sheet
    df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

# ----------------------------------------------------------------------
# CHIUSURA FILE
# ----------------------------------------------------------------------

writer.close()

print("✅ File creato:", output_file)

✅ File creato: output_scenari/Tree_Predictions_Scenarios_with_costs.xlsx


## Statistiche delle foglie

In [13]:
# -------------------------------------------------------
# INPUT
# -------------------------------------------------------

file_in = "output_scenari/Tree_Predictions_Scenarios_with_costs.xlsx"
xls = pd.ExcelFile(file_in)

all_results = []

# -------------------------------------------------------
# LOOP SU FOGLI
# -------------------------------------------------------

for sheet in xls.sheet_names:

    df = pd.read_excel(xls, sheet_name=sheet)

    # ---------------------------------------------------
    # CORRETTEZZA
    # ---------------------------------------------------

    df["correct"] = df["TREE_PREDICTION"] == df["Strategia_Ottimale"]

    # separazione errori e successi
    df["error_cost"] = df["extra_costo"].where(~df["correct"], 0)
    df["correct_saving"] = df["savings"].where(df["correct"], 0)

    # aggiungiamo anche percentuali (già presenti nel tuo file precedente)
    df["correct_saving_pct"] = df["savings_percentuale"].where(df["correct"], 0)
    df["error_cost_pct"] = df["extra_costo_percentuale"].where(~df["correct"], 0)

    # ---------------------------------------------------
    # GROUP BY FOGLIA
    # ---------------------------------------------------

    grouped = df.groupby("LEAF_ID")

    for leaf_id, g in grouped:
        leaf_pred_class = g["TREE_PREDICTION"].value_counts().idxmax()
        n = len(g)
        n_correct = g["correct"].sum()
        n_wrong = n - n_correct

        accuracy = n_correct / n if n > 0 else 0
        inaccuracy = 1 - accuracy

        # ===================================================
        # SAVING (solo corretti)
        # ===================================================

        if n_correct > 0:
            saving_mean = g.loc[g["correct"], "savings"].mean()
            saving_mean_pct = g.loc[g["correct"], "savings_percentuale"].mean()
        else:
            saving_mean = 0
            saving_mean_pct = 0

        saving_expected = saving_mean * accuracy
        saving_expected_pct = saving_mean_pct * accuracy

        # ===================================================
        # COSTO (solo errati)
        # ===================================================

        if n_wrong > 0:
            cost_mean = g.loc[~g["correct"], "error_cost"].mean()
            cost_mean_pct = g.loc[~g["correct"], "extra_costo_percentuale"].mean()
        else:
            cost_mean = 0
            cost_mean_pct = 0

        cost_expected = cost_mean * inaccuracy
        cost_expected_pct = cost_mean_pct * inaccuracy

        # ===================================================
        # SALVATAGGIO RISULTATO
        # ===================================================

        all_results.append({
            "scenario": sheet,
            "leaf_id": leaf_id,
            "leaf_predicted_class": leaf_pred_class,
            "accuracy": accuracy,
            "inaccuracy": inaccuracy,

            # -------------------------
            # SAVING ASSOLUTO
            # -------------------------
            "saving_mean": saving_mean,
            "saving_expected": saving_expected,

            # -------------------------
            # SAVING %
            # -------------------------
            "saving_mean_pct": saving_mean_pct,
            "saving_expected_pct": saving_expected_pct,

            # -------------------------
            # COSTO ASSOLUTO
            # -------------------------
            "cost_mean": cost_mean,
            "cost_expected": cost_expected,

            # -------------------------
            # COSTO %
            # -------------------------
            "cost_mean_pct": cost_mean_pct,
            "cost_expected_pct": cost_expected_pct,

            # -------------------------
            # SUPPORTO STATISTICO
            # -------------------------
            "n_samples": n
        })

# -------------------------------------------------------
# OUTPUT FINALE
# -------------------------------------------------------

df_results = pd.DataFrame(all_results)

output_file = "output_scenari/Leaf_Summary_Statistics.xlsx"

df_results.to_excel(output_file, index=False)

print("✅ Salvato:", output_file)

✅ Salvato: output_scenari/Leaf_Summary_Statistics.xlsx


## Tool interattivo

In [14]:
def query_tree(models, p, s, o, Cinter, CSystPdM, Beta, Alfa):
    key = (p, s, o)
    
    if key not in models:
        return {"error": "Scenario non trovato"}

    clf = models[key]

    X_new = pd.DataFrame([{
        "Cinter": Cinter,
        "CSystPdM": CSystPdM,
        "Beta": Beta,
        "Alfa": Alfa
    }])

    leaf = clf.apply(X_new)[0]
    pred = clf.predict(X_new)[0]

    return {
        "scenario": key,
        "strategy": pred
    }



while True:

    print("\nInserisci i valori (o scrivi 'exit' per uscire)\n")

    p = input("Predictability (HIGH/MEDIUM/LOW): ").strip().upper()
    if p.lower() == "exit":
        break

    s = input("Severity (HIGH/MEDIUM/LOW): ").strip().upper()
    if s.lower() == "exit":
        break

    o = input("Occurrence (HIGH/MEDIUM/LOW): ").strip().upper()
    if o.lower() == "exit":
        break

    Cinter = float(input("Cinter: "))
    CSystPdM = float(input("CSystPdM: "))
    Beta = float(input("Beta: "))
    Alfa = float(input("Alfa: "))

    result = query_tree(models, p, s, o, Cinter, CSystPdM, Beta, Alfa)

    print("\n📌 RESULT")
    print(result)